In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium

from folium.plugins import HeatMap, MarkerCluster
from sklearn.cluster import DBSCAN

In [ ]:
BASE_DIR = Path.cwd()


In [8]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "IndianRoads_dataset.xlsx"
if not DATA_PATH.exists():
    DATA_PATH = BASE_DIR.parent / "data" / "IndianRoads_dataset.xlsx"

In [9]:
print("Current folder:", BASE_DIR)
print("Dataset path:", DATA_PATH)
print("File found:", DATA_PATH.exists())

Current folder: c:\Users\Heena Praween\Desktop\Coding\machine_learning
Dataset path: c:\Users\Heena Praween\Desktop\Coding\machine_learning\IndianRoads_datasets.xlsx
File found: False


In [11]:
print(list(Path.cwd().glob("*.xlsx")))

[WindowsPath('c:/Users/Heena Praween/Desktop/Coding/machine_learning/accident_eda_ready.xlsx'), WindowsPath('c:/Users/Heena Praween/Desktop/Coding/machine_learning/bikesell.xlsx'), WindowsPath('c:/Users/Heena Praween/Desktop/Coding/machine_learning/IndianRoads_dataset.xlsx'), WindowsPath('c:/Users/Heena Praween/Desktop/Coding/machine_learning/online_orders.xlsx')]


In [12]:
if DATA_PATH.exists():
    df = pd.read_excel(DATA_PATH)
    print("\nDataset loaded successfully!")
    print(df.head())
else:
    print("\nERROR: Excel file not found.")
    print("Place IndianRoads_datasets.xlsx in the current folder.")


ERROR: Excel file not found.
Place IndianRoads_datasets.xlsx in the current folder.


In [14]:
df = pd.read_excel(DATA_PATH)


In [15]:
from sklearn.cluster import DBSCAN

# Convert latitude and longitude to radians
coordinates = np.radians(
    df[["latitude", "longitude"]].to_numpy()
)

# Parameters
earth_radius_km = 6371.0088
radius_km = 1.0
minimum_points = 5

model = DBSCAN(
    eps=radius_km / earth_radius_km,
    min_samples=minimum_points,
    metric="haversine",
    algorithm="ball_tree"
)

df["cluster_id"] = model.fit_predict(coordinates)

# Summarize detected clusters
cluster_summary = (
    df[df["cluster_id"] != -1]
    .groupby("cluster_id")
    .agg(
        accident_count=("cluster_id", "size"),
        center_latitude=("latitude", "mean"),
        center_longitude=("longitude", "mean")
    )
    .reset_index()
    .sort_values("accident_count", ascending=False)
)

print("DBSCAN hotspot analysis:")
display(cluster_summary.head(20))

print("Number of clusters:", len(cluster_summary))
print("Unclustered records:", (df["cluster_id"] == -1).sum())

df.to_csv("accidents_with_clusters.csv", index=False)
cluster_summary.to_csv("dbscan_hotspot_summary.csv", index=False)

DBSCAN hotspot analysis:


,cluster_id,accident_count,center_latitude,center_longitude
2,2,2577,30.698616,76.801333
3,3,2561,13.001551,80.200107
11,11,2521,22.549223,88.353174
8,8,2365,18.546672,73.849298
1,1,249,19.029780,72.800351
17,17,131,18.928233,72.889176
45,45,100,13.134459,77.580469
16,16,98,19.259776,72.878329
83,83,98,18.674284,73.971522
89,89,95,18.851925,72.899271


Number of clusters: 526
Unclustered records: 2325


In [16]:
from sklearn.neighbors import BallTree

coords_rad = np.radians(
    df[["latitude", "longitude"]].to_numpy()
)

tree = BallTree(coords_rad, metric="haversine")

radius_rad = 1.0 / earth_radius_km

df["nearby_accidents"] = tree.query_radius(
    coords_rad,
    r=radius_rad,
    count_only=True
)

density_results = df.sort_values(
    "nearby_accidents",
    ascending=False
)

print("Locations with the highest nearby-accident counts:")
display(
    density_results[
        ["latitude", "longitude", "nearby_accidents"]
    ].head(20)
)

density_results.to_csv("local_density_analysis.csv", index=False)

Locations with the highest nearby-accident counts:


,latitude,longitude,nearby_accidents
19394,30.764783,76.887720,37
12891,30.674438,76.722078,36
2575,30.763679,76.887437,36
16715,30.671122,76.727709,35
16669,30.763664,76.888624,34
198,30.650517,76.789449,34
12160,30.673102,76.729035,34
2691,30.610399,76.773670,33
12716,30.762265,76.887327,33
18430,30.671003,76.723939,33


In [ ]:
import folium

center = [
    df["latitude"].mean(),
    df["longitude"].mean()
]

accident_map = folium.Map(
    location=center,
    zoom_start=6
)

for _, row in cluster_summary.iterrows():
    folium.Marker(
        location=[
            row["center_latitude"],
            row["center_longitude"]
        ],
        popup=(
            f"Cluster: {row['cluster_id']}<br>"
            f"Accident records: {row['accident_count']}"
        ),
        tooltip=f"Cluster {row['cluster_id']}"
    ).add_to(accident_map)

folium.LayerControl().add_to(accident_map)

accident_map.save("road_accident_hotspots_map.html")

display(accident_map)

In [21]:
output_dir = Path.cwd() / "hotspot_outputs"
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / "accidents_with_clusters.csv", index=False)

cluster_summary.to_csv(
    output_dir / "dbscan_hotspot_summary.csv", index=False
)
density_results.to_csv(
    output_dir / "local_density_analysis.csv", index=False
)

accident_map.save(
    str(output_dir / "road_accident_hotspots_map.html")
)

print("Hotspot detection and analysis completed.")
print("Output folder:", output_dir.resolve())

Hotspot detection and analysis completed.
Output folder: C:\Users\Heena Praween\Desktop\Coding\machine_learning\hotspot_outputs


In [22]:
import folium
import pandas as pd
import numpy as np

# Check and prepare coordinates
df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]

if "latitude" not in df.columns and "lat" in df.columns:
    df.rename(columns={"lat": "latitude"}, inplace=True)

if "longitude" not in df.columns:
    for col in ["lon", "lng", "long"]:
        if col in df.columns:
            df.rename(columns={col: "longitude"}, inplace=True)
            break

if not {"latitude", "longitude"}.issubset(df.columns):
    raise ValueError(
        "Latitude/longitude columns not found. "
        "Check your dataset column names."
    )

df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

map_df = df.dropna(subset=["latitude", "longitude"]).copy()
map_df = map_df[
    map_df["latitude"].between(-90, 90)
    & map_df["longitude"].between(-180, 180)
]

if map_df.empty:
    raise ValueError("No valid geographic coordinates found.")

# Create map centered on accident locations
road_map = folium.Map(
    location=[
        map_df["latitude"].mean(),
        map_df["longitude"].mean()
    ],
    zoom_start=6
)

# Add accident location markers
for _, row in map_df.head(3000).iterrows():
    popup_text = (
        f"Latitude: {row['latitude']}<br>"
        f"Longitude: {row['longitude']}"
    )

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.6,
        popup=popup_text
    ).add_to(road_map)

# Save and display map
road_map.save("road_accident_locations_map.html")

display(road_map)

print("Map saved as road_accident_locations_map.html")

Map saved as road_accident_locations_map.html
